# 鳥類の系統樹を作ろう
## ミトコンドリア DNA(cytb 遺伝子)から進化の歴史を読み解く

このノートブックでは、**31種類の脊椎動物**(30種の鳥類 + 外群としてミシシッピワニ)のミトコンドリア DNA から、進化的な近さを表す **系統樹 (phylogenetic tree)** を作ります。

解析を 2 段構えで行うのがポイントです。

1. **入門編** — Python だけで完結する単純なアプローチ(切り詰めアラインメント + 近隣結合法)で大まかな結果をつかむ
2. **本格編** — 研究現場で実際に使われている **MAFFT** でアラインメントし、**IQ-TREE** で **最尤法 (Maximum Likelihood)** + ブートストラップで信頼度付きの系統樹を作る

### 学習目標

1. DNA 配列データ(FASTA形式)を扱えるようになる
2. **遺伝的距離** という考え方を理解する
3. **近隣結合法 (NJ法)** と **最尤法 (ML法)** の違いを体感する
4. **ブートストラップ** で枝の信頼度を測れることを知る
5. 作った樹を読み解き、**鳥類の進化系統** について考察する

### 系統樹とは

進化の歴史を「木」の形で表したもの。

- **枝の長さ** が遺伝的な近さを反映(短いほど近い)
- **分岐パターン** が共通祖先からの分かれ方を示す
- 末端(葉)が現存する種、内部の分岐点が「共通祖先」を表す

### なぜミトコンドリア DNA の cytb 遺伝子?

ミトコンドリア DNA は母系遺伝で、種を超えて比較しやすい性質を持っています。
中でも **cytochrome b (cytb)** は脊椎動物で約 1140 塩基 / 380 アミノ酸とほぼ長さが保存され、
種の系統解析に古くから使われてきた「定番の遺伝子」です。

> このノートブックは大学1年生の一般教養課程を想定し、専門用語は最小限にしています。
> ライセンス:配列データは NCBI のパブリックデータ、シルエット画像は PhyloPic(CC ライセンス、`birds/phylopic/phylopic_attribution.tsv` 参照)。

## 0. 必要ライブラリのインストール(初回のみ)

Python パッケージをインストールします。

- `biopython` : 配列解析と系統樹計算
- `matplotlib-fontja` : 日本語フォント

**MAFFT と IQ-TREE は別途インストールが必要です(後のセクション 12 で扱います)。**

In [ ]:
%pip install biopython pandas numpy matplotlib seaborn matplotlib-fontja

## 1. ライブラリの読み込みと日本語フォントの設定

グラフに日本語を出すために `matplotlib-fontja` を使います。
**順序が重要**:seaborn のスタイル設定を先に行い、その後に `matplotlib_fontja` を読み込みます。

In [ ]:
import sys
import shutil
import subprocess
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
import seaborn as sns

from Bio import SeqIO, AlignIO, Phylo
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

# 日本語フォントの設定(順序が重要)
sns.set_style("whitegrid")
try:
    import matplotlib_fontja  # noqa: F401
    _sans = [f for f in plt.rcParams["font.sans-serif"] if f != "IPAexGothic"]
    plt.rcParams["font.sans-serif"] = ["IPAexGothic"] + _sans
    print("matplotlib-fontja: OK")
except ImportError:
    print("matplotlib-fontja が見つかりません。上のセルでインストールしてください。")
plt.rcParams["axes.unicode_minus"] = False

# データファイルへのパス(このノートブックは ml/ から実行する想定)
BIRDS_DIR = Path("../birds")
FASTA_PATH = BIRDS_DIR / "birds_cytb.fasta"
METADATA_PATH = BIRDS_DIR / "birds_cytb_metadata.tsv"
PHYLOPIC_DIR = BIRDS_DIR / "phylopic" / "png"

# MAFFT / IQ-TREE 用の作業ディレクトリ
WORK_DIR = Path("./phylo_work")
WORK_DIR.mkdir(exist_ok=True)

for p in [FASTA_PATH, METADATA_PATH, PHYLOPIC_DIR]:
    assert p.exists(), f"見つかりません: {p}"
print(f"データ準備 OK: {BIRDS_DIR.resolve()}")
print(f"作業ディレクトリ: {WORK_DIR.resolve()}")

## 2. FASTA ファイルの読み込み

**FASTA** は配列データを表す最も基本的なテキスト形式です。

```
>Struthio_camelus|NC_002785.1|CYTB    ← ヘッダー(`>` で始まる)
ATGGCCCCCAACATTCGAAAATCG...           ← 配列本体
```

このリポジトリのヘッダーは `>属_種|NCBI アクセッション番号|遺伝子名` の3項目を `|` で区切った形式です。

In [ ]:
records = list(SeqIO.parse(FASTA_PATH, "fasta"))
print(f"読み込んだ配列数: {len(records)}")
print(f"最初のヘッダー : {records[0].id}")
print(f"最初の60塩基   : {str(records[0].seq)[:60]} ...")

# ヘッダーから「種名(属_種)」だけを取り出して ID を整理
for rec in records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""

print(f"\n整理後の ID 一覧(先頭5件): {[r.id for r in records[:5]]}")

## 3. メタデータ(和名・分類群)の読み込み

各種の和名・英名・分類群(目・系統)が `birds_cytb_metadata.tsv` にまとめられています。

- **目 (order)** : 古典的な分類単位(例: ペリカン目 Pelecaniformes)
- **系統 (clade)** : 大きな進化グループ
    - **Palaeognathae** : 古顎類 — ダチョウ、キーウィなど飛べない走鳥
    - **Galloanserae** : キジカモ類 — ニワトリ、カモなど
    - **Neoaves** : 新鳥類 — 上記以外の鳥類すべて(現生鳥類の約95%)
    - **Outgroup** : 外群(今回はミシシッピワニ)

**外群 (outgroup)** とは、解析対象から「最も離れている」と分かっている種で、系統樹の「根」を決めるのに使います。
鳥類はワニと共通祖先を持つ(両者は爬虫類のグループの一部)ことが化石記録などから分かっているので、ワニは外群として適切です。

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
# FASTA の ID(アンダースコア形式)と紐付けるキー
metadata["id"] = metadata["scientific_name"].str.replace(" ", "_")

print(f"メタデータ行数: {len(metadata)}")
print("\n各系統の種数:")
print(metadata["clade"].value_counts())
metadata.head()

## 4. 配列の長さと組成を確認

解析に入る前に、データの「健康状態」をチェックします。

- **配列長** がそろっているか? → そろっていなければ後でアラインメントが必要
- **GC 含量** (G + C の割合) → 種ごとに傾向があり、ミトコンドリアでは低めの傾向

In [ ]:
def gc_content(seq):
    s = str(seq).upper()
    return (s.count("G") + s.count("C")) / len(s) * 100

seq_stats = pd.DataFrame({
    "id": [r.id for r in records],
    "length": [len(r.seq) for r in records],
    "gc_pct": [gc_content(r.seq) for r in records],
}).merge(metadata[["id", "common_name_ja", "clade"]], on="id", how="left")

print("配列長の統計:")
print(seq_stats["length"].describe().round(1))
print(f"\n配列長の種類: {sorted(seq_stats['length'].unique())}")
seq_stats.head()

In [ ]:
# 系統別の GC 含量(箱ひげ図 + 個々の点)
fig, ax = plt.subplots(figsize=(9, 4))
order = seq_stats.groupby("clade")["gc_pct"].median().sort_values().index
sns.boxplot(data=seq_stats, x="clade", y="gc_pct", order=order,
            color="lightsteelblue", ax=ax)
sns.stripplot(data=seq_stats, x="clade", y="gc_pct", order=order,
              color="darkblue", alpha=0.6, size=4, ax=ax)
ax.set_xlabel("系統 (clade)")
ax.set_ylabel("GC 含量 (%)")
ax.set_title("cytb 遺伝子の GC 含量(系統別)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. 入門編 — シンプルなアラインメント

DNA から系統樹を作るには、**「同じ位置に同じ進化的起源の塩基を縦に揃える」** 作業が必要です。
これを **マルチプルアラインメント (multiple sequence alignment, MSA)** と呼びます。

今回のデータには都合のいい特徴があります。

- すべての配列が **ATG**(開始コドン)から始まっている
- 長さは 1140 〜 1159 塩基とほぼ同じ(脊椎動物の cytb は長さがよく保存されている)
- 違いは主に末尾(終止コドン周辺の数塩基)

そこで入門編では、簡単のため **全配列の先頭から共通の長さだけを切り出して使う** ことにします。
本格的なアラインメントは後のセクション 12 で MAFFT を使って行います。

In [ ]:
# 全配列の最小長まで切り詰める(=先頭から揃える)
min_len = min(len(r.seq) for r in records)
print(f"全配列を {min_len} 塩基に切り詰めます")

for rec in records:
    rec.seq = rec.seq[:min_len]

alignment = MultipleSeqAlignment(records)
print(f"\nアラインメント:")
print(f"  配列数 : {len(alignment)}")
print(f"  カラム数: {alignment.get_alignment_length()}")

## 6. 遺伝的距離の計算

種同士がどれだけ遺伝的に「離れている」かを数値化します。
今回は最もシンプルな **同一性距離 (identity distance)** を使います。

$$
d_{\mathrm{identity}}(A, B) = \frac{\text{異なる塩基の数}}{\text{全塩基数}}
$$

たとえば 1140 塩基中 100 ヶ所違えば $d = 100/1140 \approx 0.088$。

> 実際の研究では Kimura 2-parameter (K2P) や Jukes-Cantor (JC69) など、塩基置換の偏り(例: A↔G の方が A↔T より起こりやすい)を補正したモデルが使われます。
> 入門ではこのシンプルな距離で十分に系統関係が見えてきます。

In [ ]:
calculator = DistanceCalculator("identity")
dm = calculator.get_distance(alignment)

print(f"距離行列のサイズ: {len(dm.names)} × {len(dm.names)}")
print(f"\n例:")
print(f"  ダチョウ vs キーウィ(どちらも古顎類)")
print(f"    d = {dm['Struthio_camelus', 'Apteryx_rowi']:.4f}")
print(f"  ダチョウ vs ニワトリ(別系統)")
print(f"    d = {dm['Struthio_camelus', 'Gallus_gallus']:.4f}")
print(f"  ダチョウ vs ミシシッピワニ(外群)")
print(f"    d = {dm['Struthio_camelus', 'Alligator_mississippiensis']:.4f}")

## 7. 距離行列をヒートマップで可視化

行と列を **系統(clade)順** に並べると、近縁な種同士が「ブロック」を成して見えるはずです。

In [ ]:
names = list(dm.names)
n = len(names)
mat = np.array([[dm[a, b] for b in names] for a in names])
dist_df = pd.DataFrame(mat, index=names, columns=names)

# clade 順に並べ替え(Palaeognathae → Galloanserae → Neoaves → Outgroup)
clade_order_list = ["Palaeognathae", "Galloanserae", "Neoaves", "Outgroup"]
id_to_clade = metadata.set_index("id")["clade"].to_dict()
ordered = sorted(names, key=lambda x: (clade_order_list.index(id_to_clade.get(x, "Neoaves")), x))
dist_df = dist_df.loc[ordered, ordered]

ja_map = metadata.set_index("id")["common_name_ja"].to_dict()
labels_ja = [ja_map.get(x, x) for x in dist_df.index]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(dist_df, cmap="viridis",
            xticklabels=labels_ja, yticklabels=labels_ja,
            cbar_kws={"label": "同一性距離"}, ax=ax, square=True)
ax.set_title("31種間の遺伝的距離(系統順に並べ替え)")
plt.tight_layout()
plt.show()

**観察ポイント**

- 対角線(自分自身との比較)は距離 0 で濃い色
- 同じ系統の中(古顎類どうし、キジカモ類どうし)は距離が小さい(濃い色のブロック)
- 外群のワニは「他のすべての種から離れている」 → 右端と下端だけ明るい色

## 8. 近隣結合法 (NJ法) で系統樹を作る

**Neighbor Joining (NJ) 法** は距離行列から系統樹を作る代表的な方法です。
1987年に斎藤・根井(日本人研究者)により発表され、現在も広く使われています。

**アルゴリズムのアイデア(直感)**

1. 距離行列をもとに「**最も近い 2 種**」を選んで枝でつなぐ
2. その 2 種を 1 つのグループにまとめ、他の種からの距離を更新
3. 1〜2 を繰り返し、すべての種が 1 つの木に統合されるまで続ける

**長所**

- 計算が速い(数百種でも数秒)
- 各系統で進化速度が違っても、ある程度うまく動く

In [ ]:
constructor = DistanceTreeConstructor()
nj_tree = constructor.nj(dm)

# 外群(ワニ)で根を打ち直す → 系統樹が「鳥類 vs ワニ」の構造になる
nj_tree.root_with_outgroup("Alligator_mississippiensis")

# 内部ノードのデフォルト名(Inner1, Inner2, ...)はラベル表示時の邪魔になるので消す
for clade in nj_tree.get_nonterminals():
    clade.name = None

print(f"NJ 系統樹を作成しました")
print(f"  末端ノード(種)の数: {nj_tree.count_terminals()}")

## 9. 系統樹の描画(まずはシンプルに)

Biopython の `Phylo.draw` を使い、和名ラベル + 系統別の色 で描画します。

In [ ]:
clade_to_color = {
    "Palaeognathae": "#d95f02",   # オレンジ
    "Galloanserae": "#1b9e77",    # 緑
    "Neoaves":      "#7570b3",    # 紫
    "Outgroup":     "#444444",    # グレー
}
id_to_color = {row["id"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}
ja_to_color = {row["common_name_ja"]: clade_to_color.get(row["clade"], "#000000")
               for _, row in metadata.iterrows()}

def label_func(clade):
    if clade.is_terminal():
        return ja_map.get(clade.name, clade.name)
    return ""

fig, ax = plt.subplots(figsize=(10, 13))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Phylo.draw の軸警告を抑制
    Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=ax,
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
ax.set_title("鳥類 cytb の NJ 系統樹(外群=ミシシッピワニで根づけ)")
legend = [Patch(facecolor=c, label=cl) for cl, c in clade_to_color.items()]
ax.legend(handles=legend, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()

## 10. シルエット付きの系統樹

PhyloPic から得た各種のシルエット画像を末端に並べ、より直感的に読める樹を作ります。
シルエットは系統色で塗り分けます。

> PhyloPic ([phylopic.org](https://www.phylopic.org/)) は生物のシルエットを Creative Commons ライセンスで配布しているプロジェクトです。
> 帰属情報は `birds/phylopic/phylopic_attribution.tsv` を参照。

In [ ]:
# 系統樹を描画するための座標計算(Biopython の draw 内部処理を参考)
def get_x_positions(tree):
    depths = tree.depths()
    if not max(depths.values()):
        depths = tree.depths(unit_branch_lengths=True)
    return depths

def get_y_positions(tree):
    max_h = tree.count_terminals()
    heights = {tip: max_h - i for i, tip in enumerate(tree.get_terminals())}
    def calc(clade):
        for sub in clade:
            if sub not in heights:
                calc(sub)
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0
    if tree.root.clades:
        calc(tree.root)
    return heights

def colorize_silhouette(img, color):
    """黒シルエット PNG を指定色に塗り替える(アルファチャンネルは保持)"""
    if img.ndim != 3 or img.shape[2] != 4:
        return img
    rgb = to_rgb(color)
    colored = np.zeros_like(img)
    colored[..., 0] = rgb[0]
    colored[..., 1] = rgb[1]
    colored[..., 2] = rgb[2]
    colored[..., 3] = img[..., 3]
    return colored

def draw_tree_with_silhouettes(tree, title, xlabel="遺伝的距離", show_bootstrap=False):
    """系統樹を PhyloPic シルエット付きで描画する共通関数"""
    xpos = get_x_positions(tree)
    ypos = get_y_positions(tree)
    x_max = max(xpos.values())

    fig, ax = plt.subplots(figsize=(14, 16))

    def draw_branch(clade, x_start):
        x_here, y_here = xpos[clade], ypos[clade]
        col = id_to_color.get(clade.name, "#888") if clade.is_terminal() else "#888"
        ax.plot([x_start, x_here], [y_here, y_here], color=col, linewidth=1.6)
        if clade.clades:
            ys = [ypos[c] for c in clade.clades]
            ax.plot([x_here, x_here], [min(ys), max(ys)], color="#888", linewidth=1.6)
            for sub in clade.clades:
                draw_branch(sub, x_here)

    draw_branch(tree.root, 0)

    # 末端にシルエットと和名を配置
    for tip in tree.get_terminals():
        y, x = ypos[tip], xpos[tip]
        species = tip.name
        color = id_to_color.get(species, "#000")
        img_path = PHYLOPIC_DIR / f"{species}.png"
        if img_path.exists():
            img = colorize_silhouette(mpimg.imread(img_path), color)
            ab = AnnotationBbox(OffsetImage(img, zoom=0.02),
                                (x + x_max * 0.15, y),
                                xycoords="data", frameon=False)
            ax.add_artist(ab)
        ja = ja_map.get(species, species)
        ax.text(x + x_max * 0.01, y, ja, va="center", fontsize=10, color=color)

    # 内部ノードにブートストラップ値を表示(ある場合)
    if show_bootstrap:
        for clade in tree.get_nonterminals():
            if clade.confidence is None or clade == tree.root:
                continue
            bs = clade.confidence
            bc = "#1b9e77" if bs >= 95 else ("#d95f02" if bs >= 70 else "#b22222")
            ax.text(xpos[clade], ypos[clade], f"{bs:.0f}",
                    fontsize=8, ha="right", va="center", color=bc,
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="white",
                              edgecolor="none", alpha=0.8))

    ax.set_xlim(-x_max * 0.02, x_max * 1.30)
    ax.set_ylim(0, tree.count_terminals() + 1)
    ax.set_yticks([])
    ax.set_xlabel(xlabel)
    ax.set_title(title, fontsize=13)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)

    handles = [Patch(facecolor=c, label=cl) for cl, c in clade_to_color.items()]
    if show_bootstrap:
        handles += [
            Patch(facecolor="#1b9e77", label="BS ≥ 95"),
            Patch(facecolor="#d95f02", label="BS 70–94"),
            Patch(facecolor="#b22222", label="BS < 70"),
        ]
    ax.legend(handles=handles, loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()

draw_tree_with_silhouettes(nj_tree, "鳥類 cytb の NJ 系統樹(PhyloPic シルエット付き)")

## 11. 別の方法(UPGMA)と比較してみる

**UPGMA** は NJ よりさらにシンプルな方法で、「距離が最も近いペアを順次まとめていく」だけです。
ただし **すべての系統で進化速度が一定** という強い仮定を置くため、NJ よりも精度が落ちることが知られています。

NJ と UPGMA を並べて、どこが違うか観察してみましょう。

In [ ]:
upgma_tree = constructor.upgma(dm)
for clade in upgma_tree.get_nonterminals():
    clade.name = None

fig, axes = plt.subplots(1, 2, figsize=(20, 14))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
    axes[0].set_title("NJ (Neighbor Joining)")
    Phylo.draw(upgma_tree, label_func=label_func, do_show=False, axes=axes[1],
               label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
    axes[1].set_title("UPGMA")
plt.tight_layout()
plt.show()

**比較のポイント**

- UPGMA は **左右対称な「超距離木」** になりやすい(すべての末端が同じ深さに揃う)
- NJ は **進化速度の違い** を反映して、枝の長さがバラつく
- 大まかな分岐パターン(古顎類・キジカモ類・新鳥類の3分割)は両者で似ているはず

## 12. 本格編 — MAFFT でアラインメントし、IQ-TREE で最尤系統樹を推定

ここからは、研究現場で実際に使われている **業界標準ツール** で同じデータを解析し直します。

| ツール | 役割 | 入門編との違い |
| --- | --- | --- |
| **MAFFT** | マルチプルアラインメント | 配列を端で切るだけでなく、必要に応じて **ギャップ(挿入・欠失)** を入れて精密に揃える |
| **IQ-TREE** | 最尤法による系統樹推定 | 距離法ではなく、塩基置換モデルに基づく **確率的推論**。**ブートストラップ** で各枝の信頼度も算出 |

これらは Python ライブラリではなく **独立した実行ファイル(コマンドラインツール)** です。

### 12.1 MAFFT と IQ-TREE のインストール

環境に応じて以下のいずれかを実行してください(初回のみ)。

```bash
# Google Colab・Ubuntu/Debian
!apt-get install -y mafft iqtree

# conda 環境(より新しい IQ-TREE が手に入る)
conda install -c bioconda mafft iqtree

# macOS (Homebrew)
brew install mafft iqtree
```

> Note: 環境によって IQ-TREE のコマンド名は `iqtree`, `iqtree2`, `iqtree3` のいずれかです。
> 以下のコードでは自動検出し、バージョンに応じてオプションも切り替えます。

In [ ]:
# Colab で直接インストールしたいときは下のコメントを外す
# !apt-get install -y mafft iqtree

def find_command(candidates):
    for cmd in candidates:
        path = shutil.which(cmd)
        if path:
            return cmd, path
    return None, None

mafft_cmd, mafft_path = find_command(["mafft"])
iqtree_cmd, iqtree_path = find_command(["iqtree3", "iqtree2", "iqtree"])

print(f"MAFFT  : {mafft_path or '見つかりません'}")
print(f"IQ-TREE: {iqtree_path or '見つかりません'} (コマンド名: {iqtree_cmd})")

if not (mafft_cmd and iqtree_cmd):
    print("\n⚠ どちらかが見つからない場合は、上のセルに従ってインストールしてください。")
    print("  以下の MAFFT / IQ-TREE のセクションはスキップされ、NJ 系統樹のみ得られます。")

### 12.2 MAFFT で配列をアラインメント

**MAFFT** は速くて精度の高いマルチプルアラインメントツール(京都大学・加藤らが開発)です。
`--auto` オプションを使うと、配列数とコンピューター性能に応じた最適なアルゴリズムを自動で選んでくれます。

MAFFT に渡すために、まずヘッダーを整理した FASTA ファイルを書き出します。

In [ ]:
# 整理済み ID で FASTA を再生成(MAFFT の入力に使う)
stripped_fasta = WORK_DIR / "birds_cytb_stripped.fasta"
clean_records = list(SeqIO.parse(FASTA_PATH, "fasta"))
for rec in clean_records:
    rec.id = rec.id.split("|")[0]
    rec.description = ""
SeqIO.write(clean_records, stripped_fasta, "fasta")
print(f"ID 整理済み FASTA: {stripped_fasta}")

aligned_path = WORK_DIR / "birds_cytb_aligned.fasta"

if mafft_cmd:
    print("\nMAFFT を実行中(数秒で終わります)...")
    with open(aligned_path, "w") as f:
        subprocess.run(
            [mafft_cmd, "--auto", "--quiet", str(stripped_fasta)],
            stdout=f, stderr=subprocess.PIPE, check=True, text=True,
        )
    print(f"完了: {aligned_path}")
else:
    print("MAFFT がインストールされていません。スキップします。")

### 12.3 アラインメント結果を確認

MAFFT は挿入・欠失(ギャップ `-`)を入れて配列をきれいに揃えます。
アラインメント後のカラム数は、元のどの配列よりも長くなることが多いです。

In [ ]:
if aligned_path.exists():
    mafft_alignment = AlignIO.read(aligned_path, "fasta")
    print(f"配列数              : {len(mafft_alignment)}")
    print(f"カラム数(MAFFT後)  : {mafft_alignment.get_alignment_length()}")
    print(f"カラム数(切り詰めのみ): {alignment.get_alignment_length()}")
    diff = mafft_alignment.get_alignment_length() - alignment.get_alignment_length()
    print(f"  → MAFFT は {diff:+d} カラム分のギャップを追加")
    
    print("\n最初の60カラム × 5配列(ギャップ '-' に注目):")
    for rec in mafft_alignment[:5]:
        print(f"  {rec.id:30s} {str(rec.seq)[:60]}")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")
    mafft_alignment = None

### 12.4 保存性プロファイル(どの位置がどれだけ変わったか)

各カラムで「最も多い塩基がどれだけ多数派か」 =  **保存性スコア** を計算し、配列に沿って描きます。

- 保存率の **高い** 領域 = タンパク質の機能的に重要な部位(変えると致命的)
- 保存率の **低い** 領域 = 中立進化に近く、種ごとに多様な部位

系統解析にはこの両方が役立ちます。低保存領域 = 系統情報量が多い、高保存領域 = アラインメントの「アンカー」として機能。

In [ ]:
if mafft_alignment is not None:
    aln_array = np.array([list(str(rec.seq).upper()) for rec in mafft_alignment])
    n_seqs, n_cols = aln_array.shape
    
    conservation = np.zeros(n_cols)
    for j in range(n_cols):
        col = aln_array[:, j]
        col_no_gap = col[col != "-"]
        if len(col_no_gap) > 0:
            _, counts = np.unique(col_no_gap, return_counts=True)
            conservation[j] = counts.max() / n_seqs
    
    # 30カラムの移動平均で滑らかに
    window = 30
    smoothed = np.convolve(conservation, np.ones(window)/window, mode="valid")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(range(len(smoothed)), smoothed, color="steelblue", linewidth=1.5)
    ax.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="保存率 90%")
    ax.axhline(0.5, color="orange", linestyle="--", alpha=0.5, label="保存率 50%")
    ax.set_xlabel(f"アラインメント上のカラム位置({window}カラム移動平均)")
    ax.set_ylabel("保存性スコア")
    ax.set_title(f"cytb 遺伝子の保存性プロファイル({n_seqs}種、{n_cols}カラム)")
    ax.set_ylim(0, 1.05)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n保存率 ≥ 90% のカラム: {(conservation >= 0.9).sum()} / {n_cols} "
          f"({100*(conservation >= 0.9).mean():.1f}%)")
    print(f"保存率 < 50% のカラム: {(conservation < 0.5).sum()} / {n_cols} "
          f"({100*(conservation < 0.5).mean():.1f}%)")
else:
    print("MAFFT 結果がないため、このセルはスキップします。")

### 12.5 最尤法と IQ-TREE

**最尤法 (Maximum Likelihood, ML)** は、データを生み出す確率モデルに基づいて、
「観察された配列を最もよく説明する系統樹」を統計的に探す方法です。

**距離法 (NJ, UPGMA) との違い**

| | 距離法 (NJ) | 最尤法 (IQ-TREE) |
| --- | --- | --- |
| 入力 | 距離行列(配列の情報を要約) | アラインメント(全情報) |
| 塩基置換モデル | なし、または最初に固定 | **データから最適なモデルを自動選択**(JC, K2P, GTR + Γ + I など) |
| 計算量 | 軽い(数秒) | 重い(31種で数十秒〜2 分) |
| 信頼度 | 通常はなし | **ブートストラップ**で各枝の支持率を算出 |

**IQ-TREE のオプション**

- `-s` : 入力アラインメント
- `-m MFP` : ModelFinder Plus(最適な塩基置換モデルを自動選択)
- `-B 1000` (v2) / `-bb 1000` (v1) : UltraFast Bootstrap を 1000 回
- `-T AUTO` (v2) / `-nt AUTO` (v1) : スレッド数の自動設定
- `-redo` : 既存の出力を上書き(再実行時に便利)

ブートストラップとは、**アラインメントのカラムをランダムに重複抽出し直して系統樹を作り直す** ことを 1000 回繰り返し、
各枝が何 % の試行で再現されたかを「**支持率**」として返す仕組みです。
支持率が高い枝ほど、データの偶然のゆらぎに対して頑健であることを意味します。

In [ ]:
ml_treefile = Path(str(aligned_path) + ".treefile")

if iqtree_cmd and aligned_path.exists():
    # IQ-TREE のバージョンを検出してオプションを切り替える
    help_out = subprocess.run([iqtree_cmd, "--help"], capture_output=True, text=True)
    help_text = help_out.stdout + help_out.stderr
    is_modern = ("-B NUM" in help_text) or ("--prefix" in help_text)
    
    bootstrap_args = ["-B", "1000"] if is_modern else ["-bb", "1000"]
    threads_args = ["-T", "AUTO"] if is_modern else ["-nt", "AUTO"]
    print(f"IQ-TREE バージョン: {'v2/v3 系' if is_modern else 'v1 系'}")
    
    print(f"IQ-TREE を実行中(30秒〜2分)...")
    proc = subprocess.run(
        [iqtree_cmd, "-s", str(aligned_path), "-m", "MFP",
         *bootstrap_args, *threads_args, "-redo"],
        capture_output=True, text=True,
    )
    if proc.returncode == 0:
        print("IQ-TREE 完了")
        print("\n生成されたファイル:")
        for f in sorted(WORK_DIR.glob("birds_cytb_aligned.fasta.*")):
            print(f"  {f.name}")
    else:
        print(f"IQ-TREE がエラーで終了しました:\n{proc.stderr[-500:]}")
else:
    print("IQ-TREE またはアラインメントが利用できないため、スキップします。")

### 12.6 採用されたモデルを確認

ModelFinder は何十種類もの塩基置換モデルから AIC/BIC で最良のものを選びます。
`.iqtree` レポートファイルに採用モデルが書かれているので、それを表示してみます。

In [ ]:
report_file = Path(str(aligned_path) + ".iqtree")
if report_file.exists():
    text = report_file.read_text()
    # 採用モデル行を探して表示
    for keyword in ["Best-fit model", "Model of substitution"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
    # 対数尤度と AIC
    for keyword in ["Log-likelihood of the tree", "Unconstrained log-likelihood",
                    "Akaike information criterion (AIC) score",
                    "Bayesian information criterion (BIC) score"]:
        for line in text.splitlines():
            if keyword in line:
                print(line.strip())
                break
else:
    print("IQ-TREE レポートがないため、このセルはスキップします。")

### 12.7 ML 系統樹の読み込みと描画

IQ-TREE の出力 `.treefile`(Newick 形式)を Biopython で読み込み、外群でルート付けします。
Newick の内部ノードラベルには **ブートストラップ支持率 (0〜100)** が格納されています。

In [ ]:
if ml_treefile.exists():
    ml_tree = Phylo.read(ml_treefile, "newick")
    ml_tree.root_with_outgroup("Alligator_mississippiensis")
    
    # Newick の内部ノードラベル(=ブートストラップ値の文字列)を confidence に移す
    for clade in ml_tree.get_nonterminals():
        if clade.name:
            try:
                clade.confidence = float(clade.name)
            except ValueError:
                pass
        clade.name = None
    
    bs_values = [c.confidence for c in ml_tree.get_nonterminals() if c.confidence is not None]
    print(f"ML 系統樹を読み込みました")
    print(f"  末端: {ml_tree.count_terminals()}")
    if bs_values:
        print(f"  ブートストラップ支持率: 平均 {np.mean(bs_values):.1f}, "
              f"範囲 [{min(bs_values):.0f}, {max(bs_values):.0f}]")
        print(f"  ≥ 95(強い支持)の枝: {sum(b >= 95 for b in bs_values)} / {len(bs_values)}")
else:
    print("ML 系統樹ファイルがないため、このセルはスキップします。")
    ml_tree = None

### 12.8 ML 系統樹をブートストラップ支持率付きで描画

各内部ノードに **ブートストラップ値** を表示します。

- 🟢 緑 (≥ 95): 強い支持(その枝はほぼ確実)
- 🟠 橙 (70–94): 中程度の支持
- 🔴 赤 (< 70): 弱い支持(その枝は信頼できない)

**支持率の低い枝は、データだけからは「どっちが正しいか決められなかった」** ことを意味します。

In [ ]:
if ml_tree is not None:
    draw_tree_with_silhouettes(
        ml_tree,
        title="鳥類 cytb の ML 系統樹(IQ-TREE + UltraFast Bootstrap × 1000)",
        xlabel="塩基置換数 / サイト",
        show_bootstrap=True,
    )
else:
    print("ML 系統樹がないため、このセルはスキップします。")

### 12.9 NJ と ML を並べて比較

同じデータから 2 つの方法で系統樹を作りました。並べて見比べてみましょう。

In [ ]:
if ml_tree is not None:
    fig, axes = plt.subplots(1, 2, figsize=(20, 14))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        Phylo.draw(nj_tree, label_func=label_func, do_show=False, axes=axes[0],
                   label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
        axes[0].set_title("NJ(切り詰めアラインメント + 同一性距離)")
        Phylo.draw(ml_tree, label_func=label_func, do_show=False, axes=axes[1],
                   label_colors=lambda lbl: ja_to_color.get(lbl, "#000"))
        axes[1].set_title("ML(MAFFT アラインメント + IQ-TREE 最尤法 + UFBoot 1000)")
    plt.tight_layout()
    plt.show()
else:
    print("ML 系統樹がないため比較できません。")

**比較の観察ポイント**

- **大枠の分岐(外群、3大系統)** は両者で一致しているはず。これは信頼できるシグナル。
- **末端付近の細かい枝** は方法によって少し違うことがある。ML 樹でその枝のブートストラップが低ければ、「データだけからは決められない部分」だったということ。
- **枝の長さ** は ML のほうが情報量が多い(塩基置換モデルでの推定値)。同じ短い枝でも、「ほとんど変わっていない」のか「短期間に分かれた」のかが区別できる。

## 13. 考察 — 系統樹は何を語っているか

上で得た 2 つの樹(NJ と ML)を見比べながら、以下の問いを考えてみましょう。

### Q1. 3 つの大きなグループに分かれた?

外群(ミシシッピワニ)から先にたどると、現代の鳥類分類の主要な見解と一致する **3 つの大グループ** に分かれているはずです。

| 系統 | 例 | 特徴 |
| --- | --- | --- |
| **Palaeognathae**(古顎類) | ダチョウ、キーウィ、レア、シギダチョウ | 飛べない走鳥が多い・原始的 |
| **Galloanserae**(キジカモ類) | ニワトリ、シチメンチョウ、ウズラ、カモ、ガン | 家禽として人類に重要 |
| **Neoaves**(新鳥類) | ハト、ツバメ、フクロウ、ペンギン、ワシ など | 現生鳥類の約 **95%** がここに含まれる |

ML 樹で、この 3 分割を支える枝のブートストラップ支持率はどうなっていますか?

### Q2. 「形の似ている種」が同じグループに入っているか?

- **ペンギン**(飛べない海鳥)と **ダチョウ**(飛べない走鳥)は同じグループに入った? → **入っていないはず**
- 「飛べない」という共通点は、**収斂進化 (convergent evolution)** で別々の系統で獲得された性質です
- 同様に、**ハチドリ**(蜜を吸う小鳥)と **タイヨウチョウ**(別科の蜜食鳥)も、形だけ似て系統的には離れています

### Q3. 鳥類の "中身" の系統関係はどうか?

Neoaves の中をよく見ると…

- **ペンギン** と **アホウドリ系の海鳥**(ミナミオオフルマカモメなど) → 海鳥どうしで近い?
- **猛禽類** = ワシ(Haliaeetus)、ハヤブサ(Falco)、フクロウ(Bubo) → 全部近い?
  実は、**ハヤブサ目はワシ目より、オウム目・スズメ目に近い** ことが近年の研究で分かっています。
  今回の樹はそれを再現できているでしょうか?その枝のブートストラップ支持率は?
- **キンカチョウ**(Taeniopygia)と **シジュウカラ**(Parus) → 同じスズメ目で近い?

### Q4. 枝の長さは何を語る?

- 短い枝 → その種の cytb は祖先と似たまま(進化速度が遅い、または最近分岐した)
- 長い枝 → 多くの塩基置換が起きた(進化速度が速い、または孤立して長く別系統)

ML 樹の **「塩基置換数 / サイト」** は、その枝で起きた進化的変化の累積量を表します。

### Q5. ブートストラップ支持率が低い枝はなぜ低い?

- データ(配列)に含まれる「信号」が弱い場合 → 短期間に複数の系統が連続して分かれたなど
- アラインメントの長さが足りない場合
- 進化速度が極端に違う系統が混じる「長枝誘引 (long branch attraction)」

支持率の低い部分は **「データだけでは決められなかった」** と素直に受け止め、別の遺伝子や形態のデータで補強する必要があります。

## 14. この解析の限界と注意点

MAFFT + IQ-TREE は研究レベルの解析パイプラインですが、それでも完全ではありません。

1. **1 つの遺伝子しか使っていない**
    - cytb はミトコンドリア DNA の一部にすぎず、種の真の系統("species tree")を完全には反映しません
    - 遺伝子ごとに「遺伝子の樹」が違うことすらある(**遺伝子樹 vs 種樹** の不一致)
    - 本格的な系統研究では数百〜数千の遺伝子を組み合わせます(**系統ゲノミクス**)

2. **ミトコンドリア DNA の偏り**
    - ミトコンドリアは **母親からのみ遺伝**(雑種・交雑が見えない)
    - 核 DNA とは違う歴史を持つことがある

3. **塩基置換モデルの限界**
    - IQ-TREE は ModelFinder で最良モデルを選びますが、現実の進化が完全にモデルどおりとは限らない
    - 進化速度の急変、選択圧の変化などはモデル化が難しい

4. **長枝誘引 (Long Branch Attraction, LBA)**
    - 急速に進化した(=枝が長い)種同士が、本当は関係なくても「近い」と誤って結ばれることがある
    - 今回の解析では外群(ワニ)の枝が極端に長いため、その境界の精度には注意

## 15. 発展課題

1. **哺乳類でやってみる**
    - `BIRDS_DIR` を `Path("../mammals")` に、ファイル名を `mammals_cytb.*` に変えるだけで、ほぼ同じコードで哺乳類37種の系統樹が作れます。
    - クジラ・コウモリ・ヒト・ネズミは、系統樹上でどの位置に来るか予想して試してみよう。

2. **アラインメントの可視化**
    - MAFFT 出力を [Jalview](https://www.jalview.org/) や [AliView](https://ormbunkar.se/aliview/) で開き、保存性プロファイルで見えた「変動域」が実際にどう違うのか目で確認してみよう。

3. **モデル選択を観察する**
    - `.iqtree` レポートを開き、ModelFinder がどんなモデルを比較し、なぜそれを選んだのかを読んでみよう(AIC / BIC の表が載っている)。

4. **異なるブートストラップ法**
    - `-B 1000` (UltraFast Bootstrap, UFBoot) と `-b 100` (Standard Bootstrap) で結果がどう変わるか比較してみよう。
    - UFBoot は速いが楽観的、Standard は遅いが保守的。

5. **ベイズ法と比較する**
    - MrBayes(`apt install mrbayes`)で同じデータを解析し、事後確率(posterior probability)とブートストラップ支持率を比較してみよう。

6. **公開データと比較する**
    - [TimeTree](http://www.timetree.org/) や [OneZoom](http://www.onezoom.org/) で本物の鳥類系統樹を見てみよう。
    - 今回作った樹と、どこが一致してどこが違う?支持率の低い枝が、文献の樹とよくずれているはず。

7. **時間軸つきの樹を作る**
    - 化石記録から「ワニと鳥類の分岐は約 2.5 億年前」と仮定して、IQ-TREE の `--date` オプションや `LSD` ソフトで分子時計分析をしてみよう。